**Data preparation for RAG -> unstructured data**


In [0]:
# Installing utilities and libraries
%pip install numpy  openai  langchain-community  PyPDF2 

In [0]:
# Restart the python env
dbutils.library.restartPython()

**Uploading knowledge base docs to Unity Catalog volume and Local File system**


In [0]:
%sql
-- Create a volume
CREATE VOLUME IF NOT EXISTS dbx_apps_poc.mlpractice.genAI_lab

In [0]:
# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

print(catalog_name)

In [0]:
%run  ../../utils/upload_data
   

In [0]:
import os
import urllib.request as urllib
from urllib.parse import quote

source_folder = "https://raw.githubusercontent.com/kuljotSB/DatabricksUdemyCourse/main/GenAI/RAG/knowledge_base/"
dbfs_target_folder = f"/Volumes/{catalog_name}/mlpractice/genAI_lab/tmp/knowledge_base"

dbutils.fs.mkdirs(dbfs_target_folder)

subfolders = ['docs']

for subfolder in subfolders:
    git_subfolder = os.path.join(source_folder, subfolder)
    dbfs_subfolder = f"{dbfs_target_folder}/{subfolder}"

    dbutils.fs.mkdirs(dbfs_subfolder)

    files = [
        "Dubai Brochure.pdf",
        "Las Vegas Brochure.pdf",
        "London Brochure.pdf",
        "Margies Travel Company Info.pdf",
        "New York Brochure.pdf",
        "San Francisco Brochure.pdf"
    ]

    # ✅ Get existing files in DBFS
    existing_files = set()
    try:
        existing_files = {f.name for f in dbutils.fs.ls(dbfs_subfolder)}
    except:
        pass

    for file_name in files:
        if file_name in existing_files:
            print(f"Skipping (already exists): {file_name}")
            continue

        url = f"{git_subfolder}/{quote(file_name)}"         

        # ✅ Download only if not present
        urllib.urlretrieve(url, f"{dbfs_subfolder}/{file_name}")

         

        print(f"Uploaded: {file_name}")


Create a RAG Table schema in Unity Catalog



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbx_apps_poc.RAG


In [0]:
# Extract pdf file contents and store as a table with LangChain chunking strategy

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from PyPDF2 import PdfReader

def perform_fixed_size_chunking(text_document, chunk_size=2000, chunk_overlap= 500):
    """
    Performs recursive chunking on a document with specified overlap and chunk size. Uses RecursiveCharacterTextSplitter from langchain_text_splitters which tries multiple separators.
    """    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators = ["\n\n", "\n", " ", ""]
    )
    return text_splitter.split_text(text_document)

In [0]:
import os

dbfs_docs_folder = f"/Volumes/{catalog_name}/mlpractice/genai_lab/tmp/knowledge_base/docs"
all_docs = []

for fileName in os.listdir(dbfs_docs_folder):
    dbfs_path = os.path.join(dbfs_docs_folder, fileName)

    if os.path.isfile(dbfs_path) and fileName.lower().endswith(".pdf"):
        with open(dbfs_path, "rb") as f:
            reader = PdfReader(f)
            text = ""
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text
            if text.strip():
                chunks = perform_fixed_size_chunking(text)
                for i, chunk in enumerate(chunks):
                    if chunk.strip():
                        all_docs.append(
                            {
                                "content_path": dbfs_path,
                                "chunk": chunk
                            }
                        )


if all_docs:
    df = spark.createDataFrame(all_docs)
    print("*"*12)
    print("\n")
    print(f"Total chunks created: {df.count()}")
    print(f"\nChunks per document:")
    df.groupBy("content_path").count().show(truncate=False)
    print("*"*12)

else: 
    prin("No chunks extracted from documents")
            

In [0]:
display(df)

In [0]:
# Save the the table as a Delta table
df.write.mode("overwrite").saveAsTable("dbx_apps_poc.rag.docs_chunks")

In [0]:
# Create the final multi-modal RAG tabel in Unity Catalog
spark.sql("""
    CREATE OR REPLACE TABLE dbx_apps_poc.rag.final_rag_dataset AS
    SELECT 
        monotonically_increasing_id() as id, 
        content_path, 
        chunk 
    FROM dbx_apps_poc.rag.docs_chunks          
""")